# Email Relevance Classifier
Binary classification: predicts whether an email is **relevant (yes)** or **irrelevant (no)** to a student's interests.

**Models implemented:**
1. Logistic Regression (TF-IDF)
2. Support Vector Machine — LinearSVC (TF-IDF)
3. Naive Bayes — MultinomialNB (TF-IDF)
4. Random Forest (TF-IDF)
5. Sentence-Transformer + Logistic Regression (Semantic Embeddings)

## 1. Imports

In [1]:
import pandas as pd
import numpy as np

# Preprocessing & Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
# Sentence Transformer (Model 5)

from sentence_transformers import SentenceTransformer

print('All imports successful.')


e:\Miniconda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


## 2. Load Dataset

In [ ]:

DATASET_PATH = 'email_dataset.csv'   # Must have columns: email_text, label
# ─────────────────────────────────────────────────────────────────────────────
df2 = pd.read_csv("student_email_dataset.csv")
df = pd.read_csv(DATASET_PATH)
df2.drop(columns=['student_id','student_interests','event_category'], inplace=True)

df.drop(columns=['body'], inplace=True) 
df.rename(columns={'label': 'relevant','subject':'email_text'}, inplace=True)  # Rename the label column

df = pd.concat([df, df2], ignore_index=True)  # Combine datasets

df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


print(f'Total samples    : {len(df)}')
print(f'Training samples : {len(train_df)}')
print(f'Testing samples  : {len(test_df)}')
print('\nDataset preview:')
df.head()

Total samples    : 4000
Training samples : 3200
Testing samples  : 800

Dataset preview:


,email_text,relevant
0,University Bookstore Sale,0
1,Campus Fitness Challenge,1
2,Library Extended Hours,0
3,Annual Drama Performance,0
4,Cricket League Registration,1


## 3. Preprocessing

In [90]:
# ── Column name mapping — adjust if your columns are named differently ────────
TEXT_COL  = 'email_text'
LABEL_COL = 'relevant'
# ─────────────────────────────────────────────────────────────────────────────

X_train_raw = train_df[TEXT_COL].astype(str).tolist() # Convert to list of strings
X_test_raw  = test_df[TEXT_COL].astype(str).tolist()

print(X_train_raw)
y_train = train_df[LABEL_COL].values
y_test  = test_df[LABEL_COL].values
        # Encode labels: yes → 1, no → 0

label_map = {0: 'no', 1: 'yes'}
y_train = [label_map[v] for v in y_train]
y_test  = [label_map[v] for v in y_test]  # Manually set classes to ensure correct mapping

print('Label classes:', np.unique(label_map.values()))   # Should show: [no, yes]
print('Label distribution:\n', df[LABEL_COL].value_counts())

['Entrepreneurship Summit 2026: 2-day event with panels, workshops, and networking. Date: May 10.', 'Inter-University Football Tournament', 'CodeStorm 2026: 24-hour coding challenge open to all departments. Theme: Smart Campus. Date: May 10. Register before May 8.', 'Transport Schedule Change', 'Debate Competition Announcement', 'Mentorship Program: Get paired with successful alumni entrepreneurs. Applications close April 28.', '5K Campus Run: Timed run around campus. Date: May 2. Register by April 28. Certificate for all finishers.', 'Debate Competition Announcement', 'Public Speaking Workshop: Overcome stage fear and build arguments. Date: May 20. Venue: Seminar Hall 3. Free entry.', 'Housing Application Deadline', 'Health Awareness Session', 'Travel Club Trip Announcement', 'Annual Drama Performance', 'Film Screening Night', 'Dance Society Auditions', 'Photography Exhibition', 'Cafeteria Menu Update', 'Environment Club Tree Plantation Drive', 'Culinary Club Bake Sale', 'Photography 

## 4. TF-IDF Vectorization (Models 1–4)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # Unigrams + Bigrams
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf  = tfidf.transform(X_test_raw)

print(f'TF-IDF feature matrix shape — Train: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}')


TF-IDF feature matrix shape — Train: (3200, 1122), Test: (800, 1122)


800

## 5. Semantic Embeddings (Model 5)

In [84]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

X_train_emb = embedder.encode(X_train_raw, show_progress_bar=True, convert_to_numpy=True)
X_test_emb  = embedder.encode(X_test_raw,  show_progress_bar=True, convert_to_numpy=True)

print(f'Embedding shape — Train: {X_train_emb.shape}, Test: {X_test_emb.shape}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2066.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 25/25 [00:09<00:00,  2.64it/s]

Embedding shape — Train: (3200, 384), Test: (800, 384)


## 6. Helper — Evaluate & Display Results

In [68]:

results = {}   # Stores predictions from all models

def evaluate(model_name, model, X_tr, y_tr,X_te, y_test):
    """Trains the model, predicts on test set, and prints a training accuracy report."""
    model.fit(X_tr, y_tr)
    
    test_preds       = model.predict(X_te)
   
    test_acc   = accuracy_score(y_test, test_preds)

    results[model_name] = test_preds

    print(f'\n{'='*55}')
    print(f'  {model_name}')
    print(f'{'='*55}')
    print(f'  Test Accuracy : {test_acc:.4f}')
    print(f'\n  Test Classification Report:')
    
    print(classification_report(
    y_true=np.array(y_test), 
    y_pred=test_preds, 
    target_names=list(label_map.values())
))
   
    return model

## 7. Model 1 — Logistic Regression (TF-IDF)

In [69]:
lr_model = evaluate(
    
    model_name = 'Logistic Regression (TF-IDF)',
    model      = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    X_tr       = X_train_tfidf,
    y_tr       = y_train,
    X_te       = X_test_tfidf,
    y_test     = y_test
)


  Logistic Regression (TF-IDF)
  Test Accuracy : 0.7875

  Test Classification Report:
              precision    recall  f1-score   support

          no       0.78      0.89      0.83       472
         yes       0.80      0.64      0.71       328

    accuracy                           0.79       800
   macro avg       0.79      0.77      0.77       800
weighted avg       0.79      0.79      0.78       800



## 8. Model 2 — Support Vector Machine (TF-IDF)

In [70]:
svm_model = evaluate(
    model_name = 'Support Vector Machine — LinearSVC (TF-IDF)',
    model      = LinearSVC(C=1.0, max_iter=2000),
    X_tr       = X_train_tfidf,
    y_tr       = y_train,
    X_te       = X_test_tfidf,
    y_test     = y_test
)


  Support Vector Machine — LinearSVC (TF-IDF)
  Test Accuracy : 0.7825

  Test Classification Report:
              precision    recall  f1-score   support

          no       0.78      0.88      0.83       472
         yes       0.78      0.65      0.71       328

    accuracy                           0.78       800
   macro avg       0.78      0.76      0.77       800
weighted avg       0.78      0.78      0.78       800



## 9. Model 3 — Naive Bayes (TF-IDF)

In [71]:
nb_model = evaluate(
    model_name = 'Naive Bayes — MultinomialNB (TF-IDF)',
    model      = MultinomialNB(alpha=1.0),
    X_tr       = X_train_tfidf,
    y_tr       = y_train,
    X_te       = X_test_tfidf,
    y_test     = y_test
)


  Naive Bayes — MultinomialNB (TF-IDF)
  Test Accuracy : 0.7800

  Test Classification Report:
              precision    recall  f1-score   support

          no       0.83      0.79      0.81       472
         yes       0.72      0.76      0.74       328

    accuracy                           0.78       800
   macro avg       0.77      0.78      0.77       800
weighted avg       0.78      0.78      0.78       800



## 10. Model 4 — Random Forest (TF-IDF)

In [72]:
rf_model = evaluate(
    model_name = 'Random Forest (TF-IDF)',
    model      = RandomForestClassifier(n_estimators=200, random_state=42),
    X_tr       = X_train_tfidf,
    y_tr       = y_train,
    X_te       = X_test_tfidf,
    y_test     = y_test
)


  Random Forest (TF-IDF)
  Test Accuracy : 0.7863

  Test Classification Report:
              precision    recall  f1-score   support

          no       0.79      0.86      0.83       472
         yes       0.77      0.68      0.72       328

    accuracy                           0.79       800
   macro avg       0.78      0.77      0.77       800
weighted avg       0.79      0.79      0.78       800



## 11. Model 5 — Sentence-Transformer + Logistic Regression (Semantic Embeddings)

In [85]:
st_model = evaluate(
    model_name = 'Sentence-Transformer + Logistic Regression (Embeddings)',
    model      = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    X_tr       = X_train_emb,
    y_tr       = y_train,
    X_te       = X_test_emb,
    y_test     = y_test
)


  Sentence-Transformer + Logistic Regression (Embeddings)
  Test Accuracy : 0.7900

  Test Classification Report:
              precision    recall  f1-score   support

          no       0.78      0.90      0.84       472
         yes       0.82      0.63      0.71       328

    accuracy                           0.79       800
   macro avg       0.80      0.77      0.77       800
weighted avg       0.79      0.79      0.78       800



## 12. Final Predictions Summary

In [89]:
results["Original"] = y_test  # Add original labels for reference

# print(results['Logistic Regression (TF-IDF)'])
summary_df = pd.DataFrame(results)
# summary_df.insert(0, 'email_text', X_test_raw)

print('Predictions from all 5 models on the test set:\n')
summary_df.tail()

Predictions from all 5 models on the test set:



,Logistic Regression (TF-IDF),Support Vector Machine — LinearSVC (TF-IDF),Naive Bayes — MultinomialNB (TF-IDF),Random Forest (TF-IDF),Sentence-Transformer + Logistic Regression (Embeddings),Original
795,no,no,no,no,no,no
796,yes,yes,yes,yes,yes,yes
797,no,no,no,no,no,no
798,yes,yes,yes,yes,yes,yes
799,no,no,no,no,no,yes
